# 01 · Data Generation

## Digital Onboarding Optimization Case Study

This notebook generates a fully synthetic dataset for a senior-level Product Analytics portfolio project.

The data simulates a digital onboarding funnel where users can either submit an identity document immediately or postpone it. The central business hypothesis is that postponing document submission creates friction, increases abandonment, and generates avoidable support contacts.

**Important:** this notebook contains no real company, customer or proprietary data.

## Synthetic Data Design

The generated data includes:

- `onboarding_users.csv`: user-level funnel outcomes
- `events.csv`: event-level product tracking data
- `support_calls.csv`: support demand generated by onboarding friction

The simulation intentionally creates realistic behavioral patterns:

- Users who submit documents immediately convert at a higher rate
- Users who postpone document submission are more likely to abandon
- Support contact probability increases when users do not complete onboarding
- Acquisition channel, device and document readiness influence outcomes

In [ ]:
# Digital Onboarding Optimization - Data Generation
# This notebook generates synthetic data for a Product Analytics + Lean Six Sigma case study.

import numpy as np
import pandas as pd
from pathlib import Path

RANDOM_SEED = 42
rng = np.random.default_rng(RANDOM_SEED)

BASE_DIR = Path("..")
DATA_DIR = BASE_DIR / "data"
DATA_DIR.mkdir(parents=True, exist_ok=True)

N_USERS = 100_000
print(f"Generating synthetic onboarding dataset with {N_USERS:,} users...")

# User base
user_ids = [f"USR_{i:06d}" for i in range(1, N_USERS + 1)]

start = pd.Timestamp("2025-01-01")
end = pd.Timestamp("2025-06-30")
signup_days = rng.integers(0, (end - start).days + 1, size=N_USERS)
signup_ts = start + pd.to_timedelta(signup_days, unit="D") + pd.to_timedelta(rng.integers(0, 86400, size=N_USERS), unit="s")

channels = rng.choice(["organic", "paid_search", "referral", "partner", "social", "email"], size=N_USERS, p=[0.34, 0.23, 0.14, 0.12, 0.10, 0.07])
device = rng.choice(["android", "ios", "desktop_web"], size=N_USERS, p=[0.55, 0.34, 0.11])
region = rng.choice(["southeast", "south", "northeast", "midwest", "north"], size=N_USERS, p=[0.49, 0.18, 0.17, 0.10, 0.06])
customer_type = rng.choice(["new_customer", "existing_customer"], size=N_USERS, p=[0.72, 0.28])
account_profile = rng.choice(["solo", "small_business", "medium_business"], size=N_USERS, p=[0.52, 0.36, 0.12])

has_document_ready = rng.binomial(1, 0.62, size=N_USERS)
is_business_hours = ((pd.Series(signup_ts).dt.hour >= 8) & (pd.Series(signup_ts).dt.hour <= 20)).astype(int).to_numpy()
session_quality = rng.normal(0, 1, size=N_USERS)

logit_upload_now = (
    -1.85
    + 1.10 * has_document_ready
    + 0.35 * (device == "ios")
    - 0.20 * (device == "desktop_web")
    + 0.25 * (customer_type == "existing_customer")
    + 0.18 * is_business_hours
    + 0.22 * (channels == "referral")
    - 0.28 * (channels == "paid_search")
    + 0.30 * session_quality
)
p_upload_now = 1 / (1 + np.exp(-logit_upload_now))
upload_choice = np.where(rng.random(N_USERS) < p_upload_now, "upload_now", "upload_later")

p1_completed = rng.random(N_USERS) < np.clip(
    0.78 + 0.04 * (customer_type == "existing_customer") - 0.03 * (channels == "paid_search") + 0.02 * (device == "ios") + 0.04 * session_quality,
    0.35,
    0.95,
)

p2_reached = p1_completed & (
    rng.random(N_USERS) < np.clip(
        0.70 + 0.05 * (upload_choice == "upload_now") - 0.08 * (upload_choice == "upload_later") + 0.03 * has_document_ready + 0.03 * session_quality,
        0.25,
        0.95,
    )
)

document_submitted = p2_reached & (
    rng.random(N_USERS) < np.where(
        upload_choice == "upload_now",
        np.clip(0.63 + 0.08 * has_document_ready + 0.03 * session_quality, 0.35, 0.92),
        np.clip(0.10 + 0.08 * has_document_ready + 0.02 * session_quality, 0.02, 0.35),
    )
)

approved = document_submitted & (
    rng.random(N_USERS) < np.where(
        upload_choice == "upload_now",
        np.clip(0.78 + 0.04 * (device == "ios") - 0.03 * (channels == "paid_search"), 0.55, 0.94),
        np.clip(0.55 + 0.03 * has_document_ready - 0.04 * (channels == "paid_search"), 0.35, 0.80),
    )
)

minutes_to_upload = np.where(
    upload_choice == "upload_now",
    rng.lognormal(mean=2.1, sigma=0.55, size=N_USERS),
    rng.lognormal(mean=5.0, sigma=0.75, size=N_USERS),
)
minutes_to_upload = np.where(document_submitted, minutes_to_upload, np.nan)

users = pd.DataFrame({
    "user_id": user_ids,
    "signup_timestamp": signup_ts,
    "signup_month": pd.Series(signup_ts).dt.to_period("M").astype(str),
    "channel": channels,
    "device": device,
    "region": region,
    "customer_type": customer_type,
    "account_profile": account_profile,
    "has_document_ready": has_document_ready,
    "is_business_hours": is_business_hours,
    "upload_choice": upload_choice,
    "p1_completed": p1_completed.astype(int),
    "p2_reached": p2_reached.astype(int),
    "document_submitted": document_submitted.astype(int),
    "approved": approved.astype(int),
    "minutes_to_upload": np.round(minutes_to_upload, 2),
})

users.to_csv(DATA_DIR / "onboarding_users.csv", index=False)
users.head()


## Quality Checks

After generating the datasets, run basic validation checks:

- Number of users
- Upload-now share
- Approval rate by upload choice
- Missing values
- Event and support-call volumes

In [ ]:
users = pd.read_csv(DATA_DIR / 'onboarding_users.csv')

summary = {
    'users': len(users),
    'upload_now_share': users['upload_choice'].eq('upload_now').mean(),
    'overall_approval_rate': users['approved'].mean(),
    'approval_rate_upload_now': users.loc[users['upload_choice'].eq('upload_now'), 'approved'].mean(),
    'approval_rate_upload_later': users.loc[users['upload_choice'].eq('upload_later'), 'approved'].mean(),
}

pd.Series(summary).to_frame('value')
